# Standalone MEEP Eigenmode Solver — 1D Slab Modes

The `gsim.meep` module exposes a standalone eigenmode solver for
**1D slab modes**. This runs
MEEP locally - no cloud job required - and returns the effective index,
field profiles, and wavevectors.

**API overview:**

| Function | What it does |
|---|---|
| `solve_slab_mode(stack, wavelength)` | 1D slab mode from a layer stack |
| `mode_z_grid(stack, n_points)` | Z-axis coordinates for field arrays |
| `refractive_index_profile(stack, z_grid, wavelength)` | n(z) from the layer stack |

All solvers return a `ModeResult` with `.n_eff`, `.fields`, `.kdom`, `.n_group`, etc.

## Part 1 - Quick Start

### Setup and imports

In [ ]:
import gdsfactory as gf
import matplotlib.pyplot as plt
import numpy as np

from gsim.common.stack import get_stack
from gsim.meep import (
    mode_z_grid,
    refractive_index_profile,
    solve_slab_mode,
    solve_slab_modes,
    solve_slab_wavelength_sweep,
)

HAS_MEEP = False
try:
    import meep as mp

    HAS_MEEP = True
    print(f"MEEP {mp.__version__} ready")
except ImportError as err:
    raise SystemExit(
        "MEEP not found. Install it via conda-forge:\n"
        "    conda install -c conda-forge pymeep"
    ) from err

resolution = 64
pml_thickness = 2

gf.gpdk.PDK.activate()
stack = get_stack()

### Slab mode in 3 lines

In [ ]:
result = solve_slab_mode(
    stack=stack,
    wavelength=1.55,
    band_num=1,
    parity="NO_PARITY",
    resolution=resolution,
    pml_thickness=pml_thickness,
)

print(f"n_eff   = {result.n_eff:.6f}")
print(f"n_group = {result.n_group}")
print(f"kdom    = {[f'{k:.6f}' for k in result.kdom]}")
print(f"band    = {result.band_num}, parity = {result.parity}")

## Part 2 - Mode Results

### Understanding ModeResult

`ModeResult` is a Pydantic model containing everything the solver
extracted from MEEP:

| Attribute | Type | Description |
|---|---|---|
| `n_eff` | `float` | Effective index |
| `wavelength` | `float` | Free-space wavelength (um) |
| `frequency` | `float` | Frequency in MEEP units (1/um) |
| `fields` | `dict[str, np.ndarray]` | Complex field arrays keyed by component |
| `kdom` | `list[float]` | Dominant wavevector [kx, ky, kz] |
| `n_group` | `float \| None` | Group index (from `mode.group_velocity`) |
| `band_num` | `int` | Mode band index (1 = fundamental) |
| `parity` | `str` | Parity constraint used |

In [ ]:
z_quick = mode_z_grid(
    stack, n_points=max(round(4.22 * resolution), 1), pml_thickness=pml_thickness
)
result_fields = solve_slab_mode(
    stack=stack,
    wavelength=1.55,
    band_num=1,
    parity="NO_PARITY",
    resolution=resolution,
    field_z_grid=z_quick,
    pml_thickness=pml_thickness,
)
for comp, arr in result_fields.fields.items():
    print(f"{comp}: shape={arr.shape}, |max|={np.abs(arr).max():.6f}")

### Z-grid and refractive index profile

Use the library functions `mode_z_grid()` and
`refractive_index_profile()` to reconstruct the coordinate system and
material profile. These replace the notebook's previous hardcoded
`eps_map`.

In [ ]:
nz = max(round(2.22 * resolution), 1)  # span x resolution for default SOI stack
z_um = mode_z_grid(stack, n_points=nz, pml_thickness=pml_thickness)
n_profile = refractive_index_profile(stack, 1.55, z_grid=z_um)

print(f"Z grid: {z_um[0]:.4f} ... {z_um[-1]:.4f} um  ({len(z_um)} points)")
print(f"Index range: {n_profile.min():.4f} - {n_profile.max():.4f}")

### Mode profile with index overlay

Plot the dominant electric field component magnitude alongside the
refractive-index profile. The slab fundamental TE mode has its primary
field in `Ey`.

In [ ]:
z_slab = mode_z_grid(
    stack, n_points=max(round(4.22 * resolution), 1), pml_thickness=pml_thickness
)
result_slab = solve_slab_mode(
    stack=stack,
    wavelength=1.55,
    resolution=resolution,
    field_z_grid=z_slab,
    pml_thickness=pml_thickness,
)
n_prof = refractive_index_profile(stack, 1.55, z_grid=z_slab)

dom_comp = max(result_slab.fields, key=lambda k: np.abs(result_slab.fields[k]).max())

fig, ax = plt.subplots(figsize=(7, 4))
color_primary = "C0"

field = result_slab.fields[dom_comp]
ax.plot(z_slab, abs(field), color=color_primary, linewidth=1.5, label=f"|{dom_comp}|")
ax.plot(
    z_slab,
    field.real,
    color=color_primary,
    alpha=0.35,
    linestyle="--",
    linewidth=1,
    label=f"Re({dom_comp})",
)
ax.plot(
    z_slab,
    field.imag,
    color=color_primary,
    alpha=0.35,
    linestyle=":",
    linewidth=1,
    label=f"Im({dom_comp})",
)

ax.set_xlabel("z (um)")
ax.set_ylabel("Field amplitude (arb. units)")

ax_idx = ax.twinx()
ax_idx.plot(z_slab, n_prof, color="C3", linewidth=1.5, label="n(z)")
ax_idx.set_ylabel("Refractive index")
ax_idx.set_ylim(bottom=0.8)

ax.set_title(
    f"Mode profile - {dom_comp}  (lambda={result_slab.wavelength:.2f} um, "
    f"n_eff={result_slab.n_eff:.4f})"
)
ax.grid(True, alpha=0.2)
fig.tight_layout()

### Field component grid

All six E- and H-field components in a subplot grid.

In [ ]:
comps = sorted(
    result_slab.fields.keys(),
    key=lambda c: "ExEyEzHxHyHz".index(c) if c in "ExEyEzHxHyHz" else 99,
)
n_cols = min(3, len(comps))
n_rows = (len(comps) + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(4 * n_cols, 3 * n_rows), squeeze=False
)
for idx, comp in enumerate(comps):
    ax = axes[idx // n_cols][idx % n_cols]
    arr = result_slab.fields[comp]
    ax.plot(z_slab, np.abs(arr), linewidth=1.2, label=f"|{comp}|")
    ax.set_title(comp)
    ax.set_xlabel("z (um)")
    ax.set_ylabel("Amplitude")
    ax.grid(True, alpha=0.2)

for idx in range(len(comps), n_rows * n_cols):
    axes[idx // n_cols][idx % n_cols].set_visible(False)

fig.suptitle(
    f"Field components - slab mode  lambda={result_slab.wavelength:.2f} um",
    fontweight="bold",
)
fig.tight_layout()

## Part 3 - Advanced

### Simulation wrapper

`Simulation.solve_modes()` resolves the stack and materials internally
from the `mode_solver` configuration, then delegates to the appropriate
solver.

In [ ]:
from gsim import meep as meep_mod

sim = meep_mod.Simulation()
sim.geometry.stack = stack
sim.mode_solver(wavelengths=[1.55])

result = sim.solve_modes().results[0]
print(f"n_eff = {result.n_eff:.6f}")

### Wavelength sweep & dispersion curve

In [ ]:
wavelengths = [1.50, 1.52, 1.54, 1.55, 1.56, 1.58, 1.60]
results_sweep = solve_slab_wavelength_sweep(
    stack=stack,
    wavelengths=wavelengths,
    band_num=1,
    parity="NO_PARITY",
    resolution=resolution,
)
n_effs = [results_sweep[wl].n_eff for wl in wavelengths]
n_groups = [results_sweep[wl].n_group for wl in wavelengths]
for wl in wavelengths:
    print(
        f"  lambda = {wl:.2f} um -> n_eff = {results_sweep[wl].n_eff:.6f}, n_group = {results_sweep[wl].n_group}"
    )

fig, ax1 = plt.subplots()
(line1,) = ax1.plot(wavelengths, n_effs, "o-", label="n_eff")
ax1.set_xlabel("Wavelength (um)")
ax1.set_ylabel("n_eff")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
group_vals = [g for g in n_groups if g is not None]
if group_vals:
    (line2,) = ax2.plot(wavelengths, n_groups, "s--", color="C1", label="n_group")
    ax2.set_ylabel("n_group")
    lines = [line1, line2]
    ax1.legend(lines, [l.get_label() for l in lines], loc="upper right")
else:
    ax1.legend(loc="upper right")

ax1.set_title("Dispersion - fundamental TE mode")
fig.tight_layout()

### Multi-band modes

The symmetric SOI slab (Si 220 nm core, SiO2 cladding) at lambda=1.55 um
has V-parameter approx 1.41 - it supports a single TE mode (TE0).
Bands beyond 1 are **leaky / radiation modes**: MEEP's MPB omega-solve
can converge to them because the finite PML-bounded cell discretises
the radiation continuum.  The library logs a warning for modes with
``n_eff`` below the minimum cladding index.

In [ ]:
from gsim.common.stack.extractor import Layer, LayerStack


def _soi_slab(t_si: float) -> LayerStack:
    return LayerStack(
        layers={
            "box": Layer(
                name="box",
                gds_layer=(0, 0),
                zmin=-2.0,
                zmax=0.0,
                thickness=2.0,
                material="sio2",
                layer_type="dielectric",
            ),
            "core": Layer(
                name="core",
                gds_layer=(1, 0),
                zmin=0.0,
                zmax=t_si,
                thickness=t_si,
                material="si",
                layer_type="dielectric",
            ),
            "clad": Layer(
                name="clad",
                gds_layer=(2, 0),
                zmin=t_si,
                zmax=t_si + 2.0,
                thickness=2.0,
                material="sio2",
                layer_type="dielectric",
            ),
        }
    )


soi = _soi_slab(0.22)
n_si = 3.4777
n_sio2 = 1.4440
V_param = (2 * np.pi / 1.55) * 0.22 / 2 * np.sqrt(n_si**2 - n_sio2**2)
print(
    f"V-parameter = {V_param:.3f}  ->  {max(1, int(2 * V_param / np.pi) + 1)} guided TE mode(s)"
)

band_results = solve_slab_modes(
    stack=soi,
    wavelength=1.55,
    band_nums=list(range(1, 5)),
    parity="NO_PARITY",
    resolution=resolution,
    pml_thickness=pml_thickness,
)
for band, r in sorted(band_results.items()):
    tag = "GUIDED" if r.n_eff > n_sio2 else "LEAKY"
    print(
        f"  band {band}: n_eff={r.n_eff:.6f}  ({tag})"
        f"  n_group={r.n_group}"
        f"  kdom={[f'{k:.4f}' for k in r.kdom[:2]]}..."
    )

### Multi-band field profiles

Plotting |Ey| for each available band on the SOI slab. Only TE0
(band 1) is physically guided; higher bands may be MPB artefacts.

In [ ]:
nz_r = max(round(4.0 * resolution), 1)
zz = mode_z_grid(soi, n_points=nz_r, pml_thickness=pml_thickness)
band_fields = solve_slab_modes(
    stack=soi,
    wavelength=1.55,
    band_nums=list(range(1, 5)),
    parity="NO_PARITY",
    resolution=resolution,
    field_z_grid=zz,
    pml_thickness=pml_thickness,
)

for comp in ("Ex", "Ey", "Ez", "Hx", "Hy", "Hz"):
    fig, ax = plt.subplots(figsize=(7, 4))
    for band, r in sorted(band_fields.items()):
        if comp in r.fields:
            tag = "guided" if r.n_eff > n_sio2 else "leaky"
            ax.plot(
                zz,
                abs(r.fields[comp]),
                linewidth=1.3,
                label=f"band {band} ({tag}) n={r.n_eff:.4f}",
            )
    ax.set_xlabel("z (um)")
    ax.set_ylabel(f"|{comp}| (arb. units)")
    ax.set_title("Multi-band slab modes - SOI 220 nm")
    ax.legend(fontsize="small", loc="upper right")
    ax.grid(True, alpha=0.2)
    fig.tight_layout()

### Parity modes

MEEP's eigenmode solver supports parity constraints along Y and Z.
For the symmetric SOI slab stack, ``EVEN_Y`` selects the fundamental
TE0 mode.  ``ODD_Y`` selects the first odd-symmetry mode --
this may be a leaky/radiation mode if the waveguide is single-mode
(V < pi/2 approx 1.57).

In [ ]:
parities = ["NO_PARITY", "EVEN_Y", "ODD_Y"]
for parity in parities:
    try:
        r = solve_slab_mode(
            stack=soi,
            wavelength=1.55,
            band_num=1,
            parity=parity,
            resolution=resolution,
            pml_thickness=pml_thickness,
        )
        tag = "guided" if r.n_eff > n_sio2 else "leaky"
        print(f"  {parity:10s}: n_eff={r.n_eff:.6f}  ({tag})")
    except RuntimeError as exc:
        print(f"  {parity:10s}: not found ({exc})")

## Part 4 - Validation

### Analytical SOI slab benchmark

Validate the MEEP solver against the analytical transcendental equation
for a symmetric dielectric slab waveguide.

**Structure:** Si core (n=3.4777 @ 1.55 um) with thickness *t* = 220 nm,
SiO2 cladding (n=1.4440 @ 1.55 um).

**Transcendental equation for TE modes:**

Even modes:  $$\tan(\kappa t/2) = \frac{\gamma}{\kappa}$$
Odd modes:   $$\tan(\kappa t/2) = -\frac{\kappa}{\gamma}$$

where $$\kappa = \sqrt{n_\mathrm{core}^2 k_0^2 - \beta^2}, \quad
\gamma = \sqrt{\beta^2 - n_\mathrm{clad}^2 k_0^2}, \quad k_0 = 2\pi/\lambda$$

### Analytical solver (inline)

In [ ]:
def solve_slab_analytical(
    wavelength: float,
    n_core: float,
    n_clad: float,
    t_core: float,
    polarization: str = "TE",
) -> dict[int, float]:
    """Solve the symmetric slab transcendental equation.

    Args:
        wavelength: Free-space wavelength (um).
        n_core: Core refractive index.
        n_clad: Cladding refractive index.
        t_core: Core thickness (um).
        polarization: ``"TE"`` or ``"TM"``.

    Returns:
        Dict mapping mode number (0=fundamental) to n_eff.
    """
    k0 = 2.0 * np.pi / wavelength

    def _te_even(beta: float) -> float:
        kappa = np.sqrt(max(n_core**2 * k0**2 - beta**2, 0))
        gamma = np.sqrt(max(beta**2 - n_clad**2 * k0**2, 0))
        if kappa == 0:
            return -1e6
        return np.tan(kappa * t_core / 2) - gamma / kappa

    def _te_odd(beta: float) -> float:
        kappa = np.sqrt(max(n_core**2 * k0**2 - beta**2, 0))
        gamma = np.sqrt(max(beta**2 - n_clad**2 * k0**2, 0))
        if gamma == 0:
            return -1e6
        return np.tan(kappa * t_core / 2) + kappa / gamma

    if polarization == "TE":
        fns = [_te_even, _te_odd]
    else:
        # TM: scale by (n_core/n_clad)^2 at the interface
        def _tm_even(beta: float) -> float:
            kappa = np.sqrt(max(n_core**2 * k0**2 - beta**2, 0))
            gamma = np.sqrt(max(beta**2 - n_clad**2 * k0**2, 0))
            if kappa == 0:
                return -1e6
            return np.tan(kappa * t_core / 2) - gamma / kappa * (n_core / n_clad) ** 2

        def _tm_odd(beta: float) -> float:
            kappa = np.sqrt(max(n_core**2 * k0**2 - beta**2, 0))
            gamma = np.sqrt(max(beta**2 - n_clad**2 * k0**2, 0))
            if gamma == 0:
                return -1e6
            return np.tan(kappa * t_core / 2) + kappa / gamma * (n_clad / n_core) ** 2

        fns = [_tm_even, _tm_odd]

    beta_min = n_clad * k0
    beta_max = n_core * k0

    results: dict[int, float] = {}
    mode_idx = 0
    for fn in fns:
        # Scan beta range for sign changes
        betas = np.linspace(beta_min + 1e-4, beta_max - 1e-4, 10000)
        vals = np.array([fn(b) for b in betas])
        sign_changes = np.where(np.diff(np.signbit(vals)))[0]
        for si in sign_changes:
            b_lo = betas[si]
            b_hi = betas[min(si + 1, len(betas) - 1)]
            try:
                from scipy.optimize import bisect

                beta_root = bisect(fn, b_lo, b_hi, xtol=1e-12)
                n_eff = beta_root / k0

                # Reject spurious "roots" caused by tan(kappa*t/2) -> +/-inf
                # (pole crossing, not a genuine zero crossing)
                kappa = np.sqrt(max(n_core**2 * k0**2 - beta_root**2, 0))
                gamma = np.sqrt(max(beta_root**2 - n_clad**2 * k0**2, 0))
                if gamma <= 0 or kappa <= 0:
                    continue
                # Reject spurious "roots" where tan(kappa*t/2) -> +/-inf (pole crossing).
                # A genuine root gives fn(root) ~= 0; a pole crossing gives
                # |fn(root)| >> 0 because the sign change occurs through +/-inf,
                # not through zero.
                kappa = np.sqrt(max(n_core**2 * k0**2 - beta_root**2, 0))
                gamma = np.sqrt(max(beta_root**2 - n_clad**2 * k0**2, 0))
                if gamma <= 0 or kappa <= 0:
                    continue
                kt2 = kappa * t_core / 2.0
                # tan poles at odd multiples of pi/2: |tan| -> inf
                if abs(np.tan(kt2)) > 1e4:
                    continue

                if n_eff not in results.values():
                    results[mode_idx] = n_eff
                    mode_idx += 1
            except Exception:
                pass

    return results

In [ ]:
# Analytical benchmark parameters
lambda_bench = 1.55
n_si = 3.4777
n_sio2 = 1.4440
t_si = 0.22

analytical = solve_slab_analytical(
    wavelength=lambda_bench,
    n_core=n_si,
    n_clad=n_sio2,
    t_core=t_si,
    polarization="TE",
)
print("Analytical TE slab modes (Si 220nm / SiO2):")
for mode, n_eff in sorted(analytical.items()):
    print(f"  TE{mode}: n_eff = {n_eff:.6f}")

### Compare MEEP with analytical

In [ ]:
from gsim.common.stack.extractor import LayerStack


def _make_soi_stack(t_si: float) -> LayerStack:
    """Build a symmetric SOI slab stack: SiO2 / Si / SiO2."""
    layers = {
        "box": Layer(
            name="box",
            gds_layer=(0, 0),
            zmin=-2.0,
            zmax=0.0,
            thickness=2.0,
            material="sio2",
            layer_type="dielectric",
        ),
        "core": Layer(
            name="core",
            gds_layer=(1, 0),
            zmin=0.0,
            zmax=t_si,
            thickness=t_si,
            material="si",
            layer_type="dielectric",
        ),
        "clad": Layer(
            name="clad",
            gds_layer=(2, 0),
            zmin=t_si,
            zmax=t_si + 2.0,
            thickness=2.0,
            material="sio2",
            layer_type="dielectric",
        ),
    }
    return LayerStack(layers=layers)


soi_stack = _make_soi_stack(t_si)

# MEEP fundamental TE0
r_meep = solve_slab_mode(
    stack=soi_stack,
    wavelength=lambda_bench,
    band_num=1,
    parity="EVEN_Y",
    resolution=resolution,
    pml_thickness=pml_thickness,
)

if 0 in analytical:
    n_analytical = analytical[0]
    rel_error = abs(r_meep.n_eff - n_analytical) / n_analytical
    print(f"MEEP  TE0: n_eff = {r_meep.n_eff:.6f}")
    print(f"Anal. TE0: n_eff = {n_analytical:.6f}")
    print(f"Relative error: {rel_error:.2e}")
else:
    print("No analytical TE0 mode found.")

# Compare higher-order modes if available (single sim, reused across bands)
higher_modes = solve_slab_modes(
    stack=soi_stack,
    wavelength=lambda_bench,
    band_nums=[2, 3, 4],
    parity="NO_PARITY",
    resolution=resolution,
    pml_thickness=pml_thickness,
)
for band, r_meep_b in sorted(higher_modes.items()):
    print(f"\nMEEP  band {band}: n_eff = {r_meep_b.n_eff:.6f}")

## Part 5 - Parameter Studies

### Core thickness sweep

Vary the silicon core thickness and re-solve the slab mode. Thicker
cores -> higher confinement -> larger n_eff.

In [ ]:
thicknesses = [0.15, 0.18, 0.22, 0.25, 0.30, 0.35, 0.40]
n_eff_thick: list[float] = []

for t_si in thicknesses:
    wg_stack = _make_soi_stack(t_si)
    r = solve_slab_mode(
        stack=wg_stack,
        wavelength=1.55,
        band_num=1,
        parity="NO_PARITY",
        resolution=resolution,
        pml_thickness=pml_thickness,
    )
    n_eff_thick.append(r.n_eff)
    print(f"  t_Si = {t_si:.2f} um -> n_eff = {r.n_eff:.6f}")

fig, ax = plt.subplots()
ax.plot(thicknesses, n_eff_thick, "s-", linewidth=1.5)
ax.set_xlabel("Si core thickness (um)")
ax.set_ylabel("n_eff")
ax.set_title("n_eff vs core thickness  (lambda = 1.55 um, slab mode)")
ax.grid(True, alpha=0.3)
fig.tight_layout()

### Summary

| Feature | API |
|---|---|
| Slab modes (1D) | `solve_slab_mode(stack, wavelength=...)` |
| Simulation wrapper | `sim.mode_solver(wavelengths=[1.55]); sim.solve_modes()` |
| Z-grid utility | `mode_z_grid(stack, n_points)` |
| Index profile | `refractive_index_profile(stack, z_grid, wavelength)` |
| Field profiles | `result.fields["Ey"]` — 1D complex array along *z* |
| Dispersion sweep | Loop over wavelengths -> n_eff(lambda) |
| Group index | `result.n_group` — direct from `mode.group_velocity` |
| Multi-band | `band_num=1, 2, 3, ...` |
| Parity modes | `parity="EVEN_Y"` etc. |
| Core thickness sweep | Vary `layer.thickness` on a copy of the stack -> n_eff(t) |
| Analytical validation | Transcendental equation benchmark for SOI slab |

**Next steps:**

- Use the slab effective index as input to the variational 2D
  approximation (issue #156)
- Compare mode-solver n_eff with full FDTD frequency-domain results
- Run the same sweeps with `palace` (finite-element) for cross-validation